# formal_30_v4 deferred internal regression

Created on 2026-10-23 after `formal_30_v4_targeted` reached 7/7 development-set passes. This experiment re-runs five deferred non-passing cases from the frozen `formal_30_v3` protocol against implementation commit `9fcdee3`. It is an internal duplicate-case regression, not an independent holdout evaluation.


## Frozen inputs and configuration

Configuration: `migration_pipeline/batch_configs/formal_30_v4_deferred_regression.json`

Raw file SHA-256: `883432b67bf39d71c186d8ba625ab033cad90344c21d9f364ea9ba3a38c2fbac`

Runner canonical SHA-256: `6ddf23a789315479d7b55693b6f2023e80eb2692a766ee9f898929d5235c2a49`

All Vue paths point directly into `data/experiments/protocols/formal_30_v3_frozen_20261022/frozen`. The dry-run resolved five complex samples and new output/report/checkpoint paths under batch id `formal_30_v4_deferred_regression`.


## Fixed execution order

1. `access_matrix`: duplicate structure-only terminal failure.
2. `localization_workspace`: duplicate loop-key and structure failure.
3. `trip_itinerary_planner`: duplicate reactive-dependency and visual failure.
4. `TaskManager`: prior generation-timeout case.
5. `MonitoringDashboard`: prior generation-timeout case.

Each sample is run once without `--force`, with full pipeline state retained. Its result is frozen before any correction or deliberate rerun. The two timeout cases use the already frozen 240-second upstream and 300-second outer timeout budgets.


## Interpretation boundary

These samples were selected because they failed or timed out in v3 and therefore are known development cases. They may demonstrate regression coverage and defect remediation, but their pass rate cannot be used as independent generalization evidence. Any later generalization claim must use a separately untouched set after the v4 implementation and this protocol are frozen.


## 1. `access_matrix` result and evaluator correction

The run completed on 2026-10-23 at 15:50:31+08:00 and passed after two Qwen Plus repair rounds. Generation used 47,330 tokens, repair used 111,950 tokens, and total usage was 159,280 tokens. Final validation and visual error lists were empty. The non-blocking `migration_hints_not_landed` warning for `lastSaved` and `saveState` was retained and was not reclassified as a failure without behavioral evidence.

The recorded batch result was structure similarity 0.9766, tag similarity 1.0, text similarity 1.0, tree edit distance 2.5, and 107 Vue / 107 San nodes. Inspection showed ten repeated class differences: the Vue renderer produced `granted`, while San produced `grant-cell granted`. The frozen Vue source declares both `class="grant-cell"` and `:class="{ granted: cell.granted }"`, so the San output was correct and the residual distance was an evaluator false positive.

Before correction, the complete run was frozen at `data/experiments/snapshots/formal_30_v4_deferred_access_matrix_pass_20261023_155031`. A focused regression test first failed with actual `granted` versus expected `grant-cell granted`. The Vue lightweight renderer was then changed only to merge and deduplicate static and evaluated dynamic class values. The related regression suites passed 50 tests.

A read-only replay used the frozen v3 Vue input and the frozen final San artifact. It passed with no errors or warnings, 107/107 nodes, structure/tag/text similarity all 1.0, and tree edit distance 0. The original summary and repair history remain unchanged; the replay is recorded separately and must not be presented as the original batch metric.

Checkpoint: 1/5 completed and passed. Next sample: `localization_workspace`.


## 2. `localization_workspace` first run: automatic pass, audit failure

The first run completed on 2026-10-23 at 16:13:39+08:00 and was automatically marked passed after one Qwen Plus repair round. Generation used 39,033 tokens, repair used 46,159 tokens, and total usage was 85,192 tokens. The recorded result had 51/51 nodes, structure similarity 0.9853, tag and text similarity 1.0, and tree edit distance 0.75.

Post-run inspection rejected this as a semantic pass. The repaired San contained `class="{{{ selected: record.key === selectedKey, reviewed: record.reviewed[language] }}}"`, which retained Vue object-style class syntax inside a San interpolation. The San renderer produced `class="}"` for all three repeated entry buttons and emitted three expression warnings. Consequently the selected and reviewed row styles were not preserved.

The untouched result and implementation sources were frozen at `data/experiments/snapshots/formal_30_v4_deferred_localization_workspace_pass_with_class_defect_20261023_161648`; all 12 recorded hashes verified. This event is classified as both a migration defect and a validator false negative, not an evaluator false positive.

A focused validator test first demonstrated the false negative, and an existing Prompt-contract test was extended to demonstrate the missing instruction. The validator now reports `invalid_object_class_binding`; the repair Prompt now requires ternary string expressions and forbids `class="{{{ ... }}}"`. The related suites pass 51 tests, and revalidating the frozen defective San now fails with the intended error code.

Checkpoint distinction: the raw batch summary says 2/5 automatically completed and passed, while audit-confirmed status remains 1/5 with `localization_workspace` awaiting a deliberate `--force` rerun. The raw checkpoint is retained unchanged for traceability.


## 2a. `localization_workspace` deliberate passing rerun

After freezing the defective first run and correcting the validator and repair Prompt, the sample was deliberately rerun with `--force`. The rerun completed on 2026-10-23 at 16:30:55+08:00 and passed after one Qwen Plus repair round. Generation used 39,002 tokens, repair used 48,305 tokens, and total usage was 87,307 tokens.

The initial generated artifact already converted the Vue object class to a San string expression. The final artifact uses `(record.key === selectedKey ? 'selected' : '') + (record.reviewed[language] ? ' reviewed' : '')`; no triple-brace object interpolation remains.

The final batch result has 51/51 nodes, structure/tag/text similarity all 1.0, and tree edit distance 0. A separate direct recheck found no validation errors or warnings and no visual errors or warnings. The result is therefore accepted as an audit-confirmed semantic pass.

The passing rerun is frozen at `data/experiments/snapshots/formal_30_v4_deferred_localization_workspace_pass_20261023_163149`. The earlier automatic pass with the class defect remains independently preserved and is not overwritten. Audit-confirmed checkpoint: 2/5; next sample: `trip_itinerary_planner`.


## 3. `trip_itinerary_planner` result

The sample completed on 2026-10-23 at 16:53:33+08:00 and passed after one Qwen Plus repair round. Generation used 36,969 tokens, repair used 44,244 tokens, and total usage was 81,213 tokens.

The initial artifact failed validation because it retained a Vue `methods` wrapper, omitted prop defaults, and hid three reactive dependencies inside presentation methods. The repaired template explicitly passes `items` to `dayCount` and `conflictIds` to both `isConflict` uses. Its `s-for` item/index aliases, inline `trackby item.id`, and the two move-button `disabled` expressions remain attached to the correct elements.

The final batch result and a separate direct replay both passed with no validation or visual errors. The direct replay also had no warnings: 46/46 nodes, structure/tag/text similarity all 1.0, and tree edit distance 0. No evaluator, validator, Prompt, or pipeline correction was needed after this run.

The result is frozen at `data/experiments/snapshots/formal_30_v4_deferred_trip_itinerary_planner_pass_20261023_165740`. Audit-confirmed checkpoint: 3/5; next sample: `TaskManager`, the first prior generation-timeout case.


## 4. `TaskManager` first complete run: failed and corrected before rerun

The prior v3 generation-timeout case completed a full run on 2026-10-23 from 17:17:02 to 17:26:16+08:00. It exhausted all three Qwen Plus repair rounds and failed with `max_repair_rounds_reached`. Generation used 65,758 tokens, repair used 226,717 tokens, and total usage was 292,475 tokens. The automatic checkpoint became 4/5 completed, 3 passed, 1 failed, and 1 pending.

The final validation errors were `vue_methods_wrapper`, `missing_event_handlers`, and `missing_event_bindings`; rendering failed with `this.showNotification is not a function`. The final artifact still nested `showNotification` under a Vue-style `methods` object, so `attached()` could not call it as a San top-level method. The initial artifact and all three repair artifacts, the 5.8 MB repair history, frozen Vue input, summary, configuration, implementation sources, tests, protocol state, and Notebook state were frozen before any correction at `data/experiments/snapshots/formal_30_v4_deferred_task_manager_failed_20261023_173104`. All 23 payload hashes verified. This is a failed internal-regression result and is not overwritten or counted as a pass.

Root-cause analysis found that the Node AST bridge crashed on Vue watch entries using object-method syntax such as `handler() { ... }`. The parser silently fell back to a regex extractor that scanned the entire multi-component script instead of the root exported object. The resulting SSM combined `StatCard` name/props, another local component's data, and root template events. It also classified `showSettings = true/false` as handler method names. Separately, the validator allowed duplicate `class` attributes, which could discard static classes. These defects explain why three repair rounds received an internally inconsistent contract and did not remove the `methods` wrapper reliably.

The correction parses object-method watch handlers in the Node bridge, derives metadata from the root AST export, scopes the fallback extractor to the root exported object, classifies assignments as inline event expressions, defensively ignores non-identifier handler names, rejects `duplicate_class_attribute`, and updates generation/repair Prompt rules to require exactly one merged class attribute per element. After correction, the frozen source extracts as `TaskManager` with 18 data fields, 33 methods including `showNotification`, and watches for `tasks` and `settings`; all three settings assignments have empty handler names. The protocol regression command passed 73 tests.

Current state: correction verified locally, deliberate rerun not yet performed. The next action is to commit this checkpoint and then rerun only `TaskManager` with `--force`; that rerun must be recorded and frozen separately. `MonitoringDashboard` remains pending and must not be started until the TaskManager rerun is audited.


## 4a. `TaskManager` corrected-SSM rerun: failed and second correction

After commit `01c5827428f040787fd69fba3d70d0332eedbf60`, `TaskManager` was deliberately rerun with `--force`. It completed from 17:52:24 to 18:05:06+08:00, exhausted three Qwen Plus repair rounds, and remained failed. Generation used 90,818 tokens, repair used 322,831 tokens, and total usage was 413,649 tokens. The automatic batch checkpoint remains 4/5 completed, 3 passed, 1 failed, and 1 pending.

The corrected SSM was verified in the repair history: both metadata and script options identify `TaskManager`; it contains 18 data fields, 33 methods, no props, and watches for `tasks` and `settings`. Therefore this failure is distinct from the first run's mixed `StatCard` contract. The initial generation's duplicate class attributes were removed by the first LLM repair, showing that the new validator and Prompt rule landed.

The final validation errors were `vue_methods_wrapper`, `nested_state_mutation`, and `missing_watch_migration`. All three LLM repairs retained the top-level Vue `methods` wrapper. `moveTask` still called `splice` on the array returned directly by `this.data.get('tasks')`. The first two repaired artifacts contained both `this.watch` registrations, but the third artifact regressed and removed them. Rendering failed with `localStorage is not defined`, which was traced to environment asymmetry: the Vue renderer already supplied browser storage stubs while the San renderer did not.

Before further changes, the complete rerun was frozen at `data/experiments/snapshots/formal_30_v4_deferred_task_manager_post_ssm_fix_failed_20261023_180506`; all 23 payload hashes verified. This remains a failed result and is not merged with the first failure or counted as a pass.

The second correction adds a guarded deterministic repair for a final top-level `methods: {...}` option and gives the San renderer the same in-memory `localStorage/sessionStorage` stubs as the Vue renderer. A read-only replay of the frozen final San, with only deterministic methods unwrapping applied in memory, rendered successfully with 45 nodes and no renderer diagnostics. Validation then contained only the genuine `nested_state_mutation` and the regressed `missing_watch_migration`. The full protocol regression command passed 76 tests.

Current state: second correction verified locally, next deliberate rerun not yet performed. Commit this checkpoint and rerun only `TaskManager` with `--force`. Keep `MonitoringDashboard` pending until the result is audited and frozen.


## 4b. `TaskManager` deterministic-repair rerun: one remaining migration defect

After commit `3309c77b38c34cbb07fc1a8770cd251f2d2a641e`, `TaskManager` was deliberately rerun with `--force`. It completed from 18:39:14 to 18:49:00+08:00, exhausted three repair rounds, and remained failed. Generation used 90,876 tokens, repair used 217,314 tokens, and total usage was 308,190 tokens. The automatic batch checkpoint remains 4/5 completed, 3 passed, 1 failed, and 1 pending.

The first repair was performed by `san_methods_wrapper_normalizer`, used zero tokens, and successfully removed the final top-level Vue `methods` wrapper. Both `this.watch('tasks', ...)` and `this.watch('settings', ...)` remained in the final artifact. All three post-repair visual evaluations passed after the San storage-environment correction; the final result had 45/45 nodes, structure similarity 0.9889, tag similarity 1.0, text similarity 1.0, and tree edit distance 0.5. Thus the second correction landed as intended and neither the previous watch regression nor the renderer failure recurred.

The only final validation error was `nested_state_mutation`. In `moveTask`, the generated code assigned `const tasks = this.data.get('tasks')`, called `tasks.splice(...)`, and only then copied the already-mutated array. This is a genuine migration defect because the original reactive array is modified before `this.data.set(...)`. The complete untouched rerun was frozen before further changes at `data/experiments/snapshots/formal_30_v4_deferred_task_manager_post_deterministic_fix_failed_20261023_184900`; all 24 payload hashes verified.

The third correction adds a narrowly guarded deterministic normalizer that clones an array alias before mutation only when the alias comes directly from `this.data.get(...)`, a recognized mutating array method is used, and the same field is explicitly written back with `this.data.set(...)`. It can complete a sole clean-visual `nested_state_mutation` without an LLM call, and it also post-processes an LLM repair in the same round when other errors coexist. The post-processing action is recorded in the repair raw response.

The full test suite passed 80 tests. A read-only replay against the frozen final artifact produced no validation or visual errors, retained both watch registrations, rendered 45 San nodes, and preserved structure similarity 0.9889 and text similarity 1.0. Current state: third correction verified locally; another deliberate `TaskManager --force` rerun is pending. `MonitoringDashboard` remains blocked until that result is audited and frozen.


## 4c. `TaskManager` audit-confirmed passing rerun

After commit `064f5dbd0c3dfecb77fda42e26cd9b4b4d7e77ae`, the third deliberate `TaskManager --force` rerun completed from 18:58:32 to 19:08:05+08:00 and passed after three repair rounds. Generation used 90,802 tokens, repair used 216,837 tokens, and total usage was 307,639 tokens. The batch checkpoint is now 4/5 completed, 4 passed, 0 failed, and 1 pending.

Repair round 1 used `san_methods_wrapper_normalizer` deterministically with zero tokens. Round 2 used Qwen Plus and the traceable pipeline post-processing action `cloned_state_array_before_local_mutation`; the resulting `moveTask` starts from `const tasks = [...this.data.get('tasks')]`, so its later `splice` no longer mutates the stored array reference. Only `hidden_template_reactive_dependency` remained after round 2. Round 3 removed that final error and passed both static and visual evaluation.

A separate direct replay using the retained SSM and frozen v3 Vue input also passed: validation errors 0, visual errors 0, 45 Vue / 45 San nodes, structure similarity 0.9889, tag similarity 1.0, text similarity 1.0, and tree edit distance 0.5. The final artifact has no Vue methods wrapper, retains two watch registrations, and contains the immutable tasks clone. The sole static warning is the retained non-blocking `migration_hints_not_landed`. The full test suite passed 80 tests.

The untouched passing result is frozen at `data/experiments/snapshots/formal_30_v4_deferred_task_manager_pass_20261023_190805`; all 24 payload hashes verified. The three earlier failed TaskManager runs remain separately frozen. This is an audit-confirmed pass for a reused development case, not independent holdout evidence. Audit-confirmed checkpoint: 4/5. Next and final sample: `MonitoringDashboard`, run once without `--force`.


## 5. `MonitoringDashboard` first complete run and pending correction rerun

The first complete run used implementation commit `3a8f017fc9a7ef577fb19a77179f08bf482a6ef5` and completed from 19:20:35 to 19:31:06+08:00. It exhausted three repair rounds and failed with `max_repair_rounds_reached`. Generation used 95,267 tokens, repair used 228,044 tokens, and total usage was 323,311 tokens. The automatic batch checkpoint became 5/5 completed, 4 passed, 1 failed, and 0 pending. The final errors were `lifecycle_behavior_not_landed` and `dom_structure_similarity_low`; the final metrics were structure 0.0, tag 0.9636, text 0.9355, and tree edit distance 137.5.

The untouched failed evidence is frozen at `data/experiments/snapshots/formal_30_v4_deferred_monitoring_dashboard_failed_20261023_193106`; all 24 payload hashes were verified. This remains a failed reused-development result and is not independent holdout evidence.

Audit separated two causes. First, Vue `mounted()` calls a business helper also named `initData()`. San reserves `initData()` for data initialization, so the migration correctly inlined all seven known child calls into `inited()` and retained the remaining mounted behavior in `attached()`. The old lifecycle validator checked only for a literal `this.initData()` call and therefore emitted a false positive. Second, repair round 3 changed three valid raw directives (`s-if="activeAlerts.length"`, `s-if="showSettings"`, and `s-if="notification.show"`) into interpolation-wrapped directive values, reducing the rendered San tree from 57 nodes to 53.

The correction now accepts lifecycle helper inlining only when every known child-method call is present, with a narrow reserved-name allowance for complete `initData` inlining into San `inited()`. It rejects wrapped `s-if`/`s-else-if`/`s-for` values, deterministically unwraps only those complete directive wrappers, applies the same safe post-processing to every LLM repair result, records `unwrapped_san_directive_expression`, and adds raw-directive rules plus forbidden examples to both prompts. The full suite passed 87 tests.

Read-only replay against the frozen evidence passed. Repair round 2 now has zero validation errors and a perfect 57/57 DOM comparison: structure/tag/text 1.0 and tree edit distance 0. The raw round-3 artifact is rejected with `invalid_san_directive_interpolation`; after in-memory deterministic normalization it also validates and replays at 57/57 with structure/tag/text 1.0 and distance 0. Because this was offline replay rather than a new LLM run, it does not replace the recorded failure. Current state: correction verified locally; commit it with the frozen failure evidence, then deliberately rerun only `MonitoringDashboard` with `--force` and freeze that result separately.


## 5a. `MonitoringDashboard` post-correction rerun: automatic pass rejected by audit

After commit `ec0fa91bb2d8b7abd6878898e9573e44c040619a`, the deliberate `MonitoringDashboard --force` rerun completed from 19:55:59 to 20:03:16+08:00. The automatic pipeline passed after three repair rounds. Generation used 95,542 tokens, repair used 114,615 tokens, and total usage was 210,157 tokens. The automatic summary therefore reports 5/5 passed. Repair round 1 used `san_directive_expression_normalizer` with zero tokens, round 2 used `san_methods_wrapper_normalizer` with zero tokens, and round 3 used Qwen Plus. The recorded final comparison had 57/57 nodes, structure 0.9912, tag 1.0, text 1.0, and distance 0.5; the residual attribute difference was the two Vue-only `ref` attributes.

Code audit rejected the automatic pass. The final San defines an ordinary top-level `watch()` method whose body calls `this.watch('settings', ...)`, but neither `inited()` nor `attached()` calls that method. San does not automatically execute an arbitrary method named `watch`, so the settings watcher is never registered. The validator had only searched the entire script for the matching `this.watch(...)` text and did not enforce lifecycle placement. Audit-confirmed status therefore remains 4/5 even though the preserved automatic summary is 5/5.

Before changing the validator or Prompt, the complete rerun was frozen at `data/experiments/snapshots/formal_30_v4_deferred_monitoring_dashboard_audit_failure_20261023_200316`; all 24 payload SHA-256 values verified. This automatic-pass audit failure remains immutable and is not counted as a pass or as independent holdout evidence.

The second correction requires each source watch expression to have its `this.watch(...)` registration directly inside `inited()` or `attached()` and reports `watch_registration_not_in_lifecycle` when the text exists only elsewhere. A guarded deterministic normalizer moves only a top-level `watch()` wrapper that actually contains `this.watch(...)` into an existing lifecycle hook, removes the inert wrapper, and records `moved_watch_registration_into_lifecycle`; the same post-processing runs after every LLM repair. Generation and repair prompts now explicitly prohibit uncalled watch wrappers.

The full suite passed 92 tests. In read-only replay, the frozen final artifact is now rejected with `watch_registration_not_in_lifecycle`; in-memory normalization removes the wrapper, places the settings registration in `inited()`, and then passes validation and visual evaluation with 57/57 nodes and tag similarity 1.0. Current state: correction verified locally but no new LLM rerun has occurred. Commit this checkpoint, then deliberately rerun only `MonitoringDashboard` with `--force` and audit/freeze the result separately.
